# LSSTCam — Monthly Evolution of Visit Counts per Band (HEALPix Sky Maps) in subplots

- **Creation date**: 2026-03-10  
- **Based on**: `2026-03-10_ConsDB_LSSTCam_HEALPix_Monthly_zoommaps.ipynb`

This notebook queries ConsDB to retrieve LSSTCam visit data and visualises  
the **month-by-month evolution of visit counts** across the 6 LSST bands (*u, g, r, i, z, y*).

For each calendar month present in the dataset the notebook produces:
1. **A 2×3 panel of HEALPix sky maps** (top row: u, g, r — bottom row: i, z, y) with band-specific
   colormaps, Galactic plane trace, and DDF `+` markers (no text labels).
2. **A temporal-evolution grid** (section 7): one all-band combined sky map per month,
   laid out left-to-right chronologically so the survey strategy becomes immediately visible.
3. A **stacked / grouped bar chart** summarising total visit counts per band per month.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import healpy as hp

from astropy.table import join
from astropy.coordinates import SkyCoord
import astropy.units as u

from lsst.summit.utils import ConsDbClient

%matplotlib inline

print(f'healpy  version : {hp.__version__}')
print(f'numpy   version : {np.__version__}')
print(f'pandas  version : {pd.__version__}')

## 2. Configuration

In [ ]:
# ── Instrument ──────────────────────────────────────────────────────────────
instrument = 'LSSTCam'

# ── HEALPix resolution ──────────────────────────────────────────────────────
# NSIDE=64  -> pixel size ~ 0.92 deg  (good overview, fast)
# NSIDE=128 -> pixel size ~ 0.46 deg  (finer, still manageable)
NSIDE = 32
NPIX  = hp.nside2npix(NSIDE)
print(f'NSIDE={NSIDE}  ->  {NPIX} pixels,  pixel size ~ {np.degrees(hp.nside2resol(NSIDE)):.2f} deg')

# ── Bands ────────────────────────────────────────────────────────────────────
BANDS = list('ugrizy')
BAND_COLORS = {
    'u': '#9b59b6',   # purple
    'g': '#2ecc71',   # green
    'r': '#e74c3c',   # red
    'i': '#e67e22',   # orange
    'z': '#3498db',   # blue
    'y': '#795548',   # brown
}

# ── Matplotlib defaults ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize' : (12, 6),
    'axes.labelsize' : 'large',
    'axes.titlesize' : 'large',
    'xtick.labelsize': 'large',
    'ytick.labelsize': 'large',
})

# ── Deep Drilling Fields (ICRS degrees) ─────────────────────────────────────
DDF_NAMES   = ['XMM-LSS', 'COSMOS', 'ECDFS', 'ELAIS-S1', 'EDFS_a', 'EDFS_b']
DDF_RA_DEG  = np.array([35.57, 150.11, 52.98,   9.45,  58.90,  63.60])
DDF_DEC_DEG = np.array([-4.82,   2.23, -28.12, -44.02, -49.32, -47.60])

print('Configuration done.')

## 3. ConsDB connection & data retrieval

In [ ]:
os.environ['no_proxy'] += ',.consdb'
url    = 'http://consdb-pq.consdb:8080/consdb'
consdb = ConsDbClient(url)
print('ConsDB client created.')

In [ ]:
# Retrieve visit data from ConsDB (same queries as parent notebook)
visits    = consdb.query('SELECT * FROM cdb_lsstcam.visit1 WHERE day_obs >= 20250415')
visits_ql = consdb.query('SELECT * FROM cdb_lsstcam.visit1')

merged_visits = join(visits, visits_ql, keys='visit_id', join_type='inner')

df_visits = visits.to_pandas()
print(f'Visits retrieved: {len(df_visits)}')

## 4. Data cleaning & month extraction

In [ ]:
# Remove engineering / pinhole filters (same logic as parent notebook)
bad_filters = {'other', 'none', 'other:pinhole'}
df_visits = df_visits[~df_visits['physical_filter'].isin(bad_filters)]

# Drop rows with missing pointing coordinates or band information
df_visits = df_visits.dropna(subset=['s_ra', 's_dec', 'band']).reset_index(drop=True)

print(f'Visits after cleaning : {len(df_visits)}')
print(f'Bands present         : {sorted(df_visits["band"].dropna().unique())}')

In [ ]:
# ── Parse observation date and extract year-month ────────────────────────────
# `day_obs` is an integer in the form YYYYMMDD (e.g. 20250415)
df_visits['day_obs_str'] = df_visits['day_obs'].astype(str)
df_visits['obs_date']    = pd.to_datetime(df_visits['day_obs_str'], format='%Y%m%d', errors='coerce')
df_visits['year_month']  = df_visits['obs_date'].dt.to_period('M')   # e.g. 2025-04

# Drop rows where the date could not be parsed
df_visits = df_visits.dropna(subset=['obs_date']).reset_index(drop=True)

# Sorted list of unique months present in the dataset
months = sorted(df_visits['year_month'].dropna().unique())

print(f'Date range : {df_visits["obs_date"].min().date()}  ->  {df_visits["obs_date"].max().date()}')
print(f'Months found ({len(months)}) : {[str(m) for m in months]}')

## 5. Helper functions

In [ ]:
def visits_to_healpix_map(ra_deg, dec_deg, nside=NSIDE):
    """
    Build a HEALPix visit-count map from pointing coordinates.

    Parameters
    ----------
    ra_deg, dec_deg : array-like  (degrees, ICRS)
    nside           : int         HEALPix NSIDE parameter

    Returns
    -------
    hpmap : np.ndarray, shape (hp.nside2npix(nside),)
        Visit count per pixel; unobserved pixels are set to hp.UNSEEN.
    """
    ra  = np.asarray(ra_deg,  dtype=float)
    dec = np.asarray(dec_deg, dtype=float)
    theta = np.radians(90.0 - dec)   # colatitude
    phi   = np.radians(ra)
    ipix  = hp.ang2pix(nside, theta, phi)
    hpmap = np.zeros(hp.nside2npix(nside), dtype=float)
    np.add.at(hpmap, ipix, 1)
    hpmap[hpmap == 0] = hp.UNSEEN
    return hpmap


def galactic_plane_icrs():
    """
    Return (ra_deg, dec_deg) arrays tracing the Galactic plane in ICRS,
    sorted by RA to avoid spurious lines at the 0/360 deg wrap.
    """
    gal_l = np.linspace(0., 360., 1440)
    gal_b = np.zeros(1440)
    gp    = SkyCoord(l=gal_l * u.deg, b=gal_b * u.deg, frame='galactic').icrs
    idx   = np.argsort(gp.ra.deg)
    return gp.ra.deg[idx], gp.dec.deg[idx]


# Pre-compute the Galactic plane trace once
GP_RA_DEG, GP_DEC_DEG = galactic_plane_icrs()

print('Helper functions defined.')

In [ ]:
# ── Band-specific colormaps ───────────────────────────────────────────────────
# Each colormap is chosen to match the physical colour of the passband so that
# the visualisation is immediately interpretable.  Sequential colormaps are used
# (light = few visits, deep/saturated = many visits).
#
#  Band   lambda_central   Colormap   Rationale
#  ----   --------------   --------   ---------
#  u        ~350 nm        Purples    UV / violet
#  g        ~475 nm        Greens     Green
#  r        ~635 nm        Reds       Red
#  i        ~775 nm        Oranges    Warm near-IR
#  z        ~925 nm        Blues      Cool near-IR (contrasts with i and y)
#  y       ~1000 nm        YlOrBr     Deep near-IR / brown-amber tones
BAND_CMAPS = {
    'u': 'Purples',
    'g': 'Greens',
    'r': 'Reds',
    'i': 'Oranges',
    'z': 'Blues',
    'y': 'YlOrBr',
}

# 2x3 panel row layout
BAND_ROWS = [['u', 'g', 'r'], ['i', 'z', 'y']]


def plot_monthly_6bands_panel_original(month_str, hpmaps_dict, nside=NSIDE):
    """
    Produce a 2x3 figure for a single month.

    Layout
    ------
        top row    :  u  |  g  |  r
        bottom row :  i  |  z  |  y

    Each subplot uses BAND_CMAPS[band].  Bands with no visits show an empty
    (all hp.UNSEEN) map so the 2x3 grid is always complete.
    The Galactic plane (grey dots) and DDF '+' markers are overlaid;
    DDF text labels are intentionally omitted for readability in compact panels.

    Parameters
    ----------
    month_str   : str   e.g. '2025-06'
    hpmaps_dict : dict  {band: hpmap_array}  (missing keys -> empty map)
    """
    empty_map = np.full(hp.nside2npix(nside), hp.UNSEEN)

    # Pre-compute per-band subtitle strings
    subtitles = {}
    for band in BANDS:
        hpmap = hpmaps_dict.get(band, empty_map)
        if np.all(hpmap == hp.UNSEEN):
            subtitles[band] = f'Band {band}  —  no data'
        else:
            n_vis = int((hpmap[hpmap != hp.UNSEEN]).sum())
            subtitles[band] = f'Band {band}   ({n_vis:,} visits)'

    fig = plt.figure(figsize=(18, 8))
    fig.suptitle(
        f'{instrument}  \u2014  {month_str}  \u2014  HEALPix visit counts  (NSIDE={nside})',
        fontsize=13, fontweight='bold', y=1.01,
    )

    for row_idx, row_bands in enumerate(BAND_ROWS):
        for col_idx, band in enumerate(row_bands):
            sp = row_idx * 3 + col_idx + 1   # subplot index 1..6
            hpmap = hpmaps_dict.get(band, empty_map)

            hp.mollview(
                hpmap,
                title=subtitles[band],
                cmap=BAND_CMAPS[band],
                unit='',
                bgcolor='white',
                badcolor='#eeeeee',
                flip='astro',
                coord='C',
                sub=(2, 3, sp),
                margins=(0.005, 0.005, 0.005, 0.005),
            )
            hp.graticule(dpar=30, dmer=60, alpha=0.18, lw=0.3)
            # Galactic plane — thin grey trace
            hp.projplot(GP_RA_DEG, GP_DEC_DEG, ',', ms=0.7,
                        color='dimgrey', alpha=0.45, lonlat=True, coord='C')
            # DDF '+' markers only — no text annotation
            hp.projscatter(DDF_RA_DEG, DDF_DEC_DEG,
                           marker='+', s=90, c='black', lw=0.7,
                           zorder=10, lonlat=True, coord='C')

    plt.tight_layout()
    plt.show()



def plot_monthly_6bands_panel(month_str, hpmaps_dict, nside=NSIDE, unit='N visits', log_scale = True):
    """
    Produce a 2x3 figure for a single month.

    Layout
    ------
        top row    :  u  |  g  |  r
        bottom row :  i  |  z  |  y

    Each subplot uses BAND_CMAPS[band].  Bands with no visits show an empty
    (all hp.UNSEEN) map so the 2x3 grid is always complete.
    The Galactic plane (grey dots) and DDF '+' markers are overlaid;
    DDF text labels are intentionally omitted for readability in compact panels.

    Parameters
    ----------
    month_str   : str   e.g. '2025-06'
    hpmaps_dict : dict  {band: hpmap_array}  (missing keys -> empty map)
    """
    empty_map = np.full(hp.nside2npix(nside), hp.UNSEEN)

    # Pre-compute per-band subtitle strings
    subtitles = {}
    for band in BANDS:
        hpmap = hpmaps_dict.get(band, empty_map)
        if np.all(hpmap == hp.UNSEEN):
            subtitles[band] = f'Band {band}  —  no data'
        else:
            n_vis = int((hpmap[hpmap != hp.UNSEEN]).sum())
            subtitles[band] = f'Band {band}   ({n_vis:,} visits)'

    fig = plt.figure(figsize=(18, 8))
    fig.suptitle(
        f'{instrument}  \u2014  {month_str}  \u2014  HEALPix visit counts  (NSIDE={nside})',
        fontsize=13, fontweight='bold', y=1.01,
    )

    for row_idx, row_bands in enumerate(BAND_ROWS):
        for col_idx, band in enumerate(row_bands):
            sp = row_idx * 3 + col_idx + 1   # subplot index 1..6
            hpmap = hpmaps_dict.get(band, empty_map)

            if log_scale:
                hp.mollview(
                    hpmap,
                    title=subtitles[band],
                    cmap=BAND_CMAPS[band],
                    norm='log',      # <--- Very important to see low-density areas
                    min=1,           # <--- Avoids log(0) issues
                    max=np.max(hpmap) if np.any(hpmap > 0) else 10,
                    unit='log10(1+N)',
                    bgcolor='white',
                    badcolor='#eeeeee',
                    flip='astro',
                    coord='C',
                    sub=(2, 3, sp),
                    margins=(0.005, 0.005, 0.005, 0.005),
                )
            else:
                hp.mollview(
                    hpmap,
                    title=subtitles[band],
                    cmap=BAND_CMAPS[band],
                    unit=unit,
                    bgcolor='white',
                    badcolor='#eeeeee',
                    flip='astro',
                    coord='C',
                    sub=(2, 3, sp),
                    margins=(0.005, 0.005, 0.005, 0.005),
                )
                
            hp.graticule(dpar=30, dmer=60, alpha=0.18, lw=0.3)
            # Galactic plane — thin grey trace
            hp.projplot(GP_RA_DEG, GP_DEC_DEG, ',', ms=0.7,
                        color='dimgrey', alpha=0.45, lonlat=True, coord='C')
            # DDF '+' markers only — no text annotation
            hp.projscatter(DDF_RA_DEG, DDF_DEC_DEG,
                           marker='+', s=90, c='black', lw=0.7,
                           zorder=10, lonlat=True, coord='C')

    plt.tight_layout()
    plt.show()


print('Band cmaps and panel function defined.')

## 6. Monthly HEALPix sky maps — 2×3 band panel per month

For every calendar month in the dataset a single figure with **6 subplots** is produced:

```
top row    :  u (Purples)  |  g (Greens)  |  r (Reds)
bottom row :  i (Oranges)  |  z (Blues)   |  y (YlOrBr)
```

Each band uses a colormap that matches its physical colour.  
The Galactic plane (grey dots) and DDF `+` markers are overlaid.  
DDF text labels are omitted to keep the compact panels readable.

In [ ]:
# Dictionary: monthly_counts[month_str][band] = n_visits  (reused by bar plots)
monthly_counts = {}
# All-bands combined HEALPix maps stored for the temporal-evolution panel (section 7)
monthly_hpmaps_allbands = {}   # {month_str: hpmap}

for month in months:
    month_str = str(month)
    df_month  = df_visits[df_visits['year_month'] == month]

    print('=' * 72)
    print(f'  Month : {month_str}   |   Total visits this month : {len(df_month):,}')
    print('=' * 72)

    monthly_counts[month_str] = {}
    hpmaps_this_month = {}     # {band: hpmap} — filled below

    # ── Build one HEALPix map per band ──────────────────────────────────────
    for band in BANDS:
        df_mb = df_month[df_month['band'] == band]
        n_vis = len(df_mb)
        monthly_counts[month_str][band] = n_vis

        if n_vis == 0:
            print(f'    Band {band}: no visits.')
            continue

        hpmap_mb = visits_to_healpix_map(
            df_mb['s_ra'].values, df_mb['s_dec'].values, nside=NSIDE)
        hpmaps_this_month[band] = hpmap_mb

        valid    = hpmap_mb[hpmap_mb != hp.UNSEEN]
        n_pix    = len(valid)
        sky_area = n_pix * hp.nside2pixarea(NSIDE, degrees=True)
        print(f'    Band {band}: {n_vis:5,} visits | '
              f'{n_pix:4d} pixels | '
              f'sky {sky_area:.0f} deg\u00b2 | '
              f'max {int(valid.max())} vis/pix')

    # ── All-bands combined map — stored for section 7 ───────────────────────
    if len(df_month) > 0:
        monthly_hpmaps_allbands[month_str] = visits_to_healpix_map(
            df_month['s_ra'].values, df_month['s_dec'].values, nside=NSIDE)
    else:
        monthly_hpmaps_allbands[month_str] = np.full(hp.nside2npix(NSIDE), hp.UNSEEN)

    # ── 2x3 panel: u,g,r / i,z,y ────────────────────────────────────────────
    if hpmaps_this_month:
        plot_monthly_6bands_panel(month_str, hpmaps_this_month, nside=NSIDE)

    print()

## 7. Temporal evolution — all-bands combined sky maps, month by month

One subplot per calendar month shows the **all-band combined HEALPix visit-count map**
(all 6 filters summed), coloured with the `YlOrRd` sequential colormap.

The grid adapts automatically to the number of months (up to 4 columns).  
Reading left-to-right / top-to-bottom reveals how the Rubin survey footprint
grows over time: the WFD tiling fills the extragalactic sky while the DDFs
(`+` markers in navy) accumulate depth month after month.

In [ ]:
# ── Temporal evolution — all-bands combined, one subplot per month ────────────

n_months = len(months)
if n_months == 0:
    print('No months available.')
else:
    # Layout: up to 4 columns, number of rows computed automatically
    ncols = min(4, n_months)
    nrows = int(np.ceil(n_months / ncols))

    fig = plt.figure(figsize=(ncols * 5.5, nrows * 3.4))
    fig.suptitle(
        f'{instrument}  \u2014  All-bands combined  \u2014  Monthly evolution  (NSIDE={NSIDE})',
        fontsize=14, fontweight='bold', y=1.01,
    )

    for m_idx, month in enumerate(months):
        month_str   = str(month)
        hpmap_month = monthly_hpmaps_allbands[month_str]
        n_vis_month = len(df_visits[df_visits['year_month'] == month])
        sp          = m_idx + 1

        hp.mollview(
            hpmap_month,
            title=f'{month_str}\n({n_vis_month:,} visits)',
            cmap='YlOrRd',
            norm='log',      # <--- Very important to see low-density areas
            min=1,           # <--- Avoids log(0) issues
            max=np.max(hpmap_month) if np.any(hpmap_month > 0) else 10,
            unit='',
            bgcolor='white',
            badcolor='#eeeeee',
            flip='astro',
            coord='C',
            sub=(nrows, ncols, sp),
            margins=(0.005, 0.005, 0.005, 0.005),
        )
        hp.graticule(dpar=30, dmer=60, alpha=0.18, lw=0.3)
        # Galactic plane — thin grey trace
        hp.projplot(GP_RA_DEG, GP_DEC_DEG, ',', ms=0.7,
                    color='dimgrey', alpha=0.45, lonlat=True, coord='C')
        # DDF '+' markers in navy for contrast against the YlOrRd colormap
        hp.projscatter(DDF_RA_DEG, DDF_DEC_DEG,
                       marker='+', s=80, c='navy', lw=0.8,
                       zorder=10, lonlat=True, coord='C')

    plt.tight_layout()
    plt.show()

## 8. Monthly visit counts — summary table

In [ ]:
# Build a tidy DataFrame from the counts collected in the loop above
df_monthly = (
    pd.DataFrame(monthly_counts).T      # rows = months, columns = bands
      .fillna(0)
      .astype(int)
)
df_monthly.index.name = 'Month'

# Ensure all 6 bands appear as columns even if some are missing
for b in BANDS:
    if b not in df_monthly.columns:
        df_monthly[b] = 0
df_monthly = df_monthly[BANDS]   # canonical band order

# Add a 'Total' column
df_monthly['Total'] = df_monthly[BANDS].sum(axis=1)

print('Monthly visit counts per band:')
display(df_monthly)

## 9. Final bar plot — monthly visit counts with all 6 bands

Each bar represents a calendar month; colours show the contribution  
from each of the 6 LSST bands (*u, g, r, i, z, y*).

In [ ]:
fig, ax = plt.subplots(figsize=(max(10, len(months) * 1.6), 6))

month_labels = [str(m) for m in months]
x = np.arange(len(month_labels))
bar_width = 0.12   # width of individual band bars
n_bands   = len(BANDS)

# Offset each band bar so that the group is centred on each month tick
offsets = np.linspace(
    -(n_bands - 1) / 2 * bar_width,
     (n_bands - 1) / 2 * bar_width,
    n_bands,
)

for band, offset in zip(BANDS, offsets):
    counts = [monthly_counts.get(m, {}).get(band, 0) for m in month_labels]
    ax.bar(
        x + offset,
        counts,
        width=bar_width,
        label=f'Band {band}',
        color=BAND_COLORS[band],
        edgecolor='white',
        linewidth=0.5,
        alpha=0.92,
    )

# Annotate the total visits above each month group
for i, m in enumerate(month_labels):
    total = df_monthly.loc[m, 'Total'] if m in df_monthly.index else 0
    if total > 0:
        ax.text(
            x[i], total * 0.4,
            f'{total:,}',
            ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='black',
        )

ax.set_xticks(x)
ax.set_xticklabels(month_labels, rotation=30, ha='right', fontsize=11)
ax.set_xlabel('Month', fontsize=13)
ax.set_ylabel('Number of visits', fontsize=13)
ax.set_title(
    f'{instrument} \u2014 Monthly visit counts per LSST band',
    fontsize=14, fontweight='bold',
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda val, _: f'{int(val):,}'))
ax.legend(
    title='Band', title_fontsize=10,
    fontsize=10, loc='upper left',
    framealpha=0.85,
)
ax.grid(axis='y', alpha=0.35, linestyle='--')
ax.set_xlim(x[0] - 0.5, x[-1] + 0.5)

plt.tight_layout()
plt.show()

## 10. Stacked bar plot — cumulative view

In [ ]:
fig, ax = plt.subplots(figsize=(max(10, len(months) * 1.6), 6))

bottom = np.zeros(len(month_labels))

for band in BANDS:
    counts = np.array([monthly_counts.get(m, {}).get(band, 0) for m in month_labels],
                      dtype=float)
    ax.bar(
        x,
        counts,
        bottom=bottom,
        label=f'Band {band}',
        color=BAND_COLORS[band],
        edgecolor='white',
        linewidth=0.5,
        alpha=0.92,
    )
    bottom += counts

# Annotate total on top of each stacked bar
for i, (m, tot) in enumerate(zip(month_labels, bottom)):
    if tot > 0:
        ax.text(
            x[i], tot * 1.012,
            f'{int(tot):,}',
            ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='black',
        )

ax.set_xticks(x)
ax.set_xticklabels(month_labels, rotation=30, ha='right', fontsize=11)
ax.set_xlabel('Month', fontsize=13)
ax.set_ylabel('Number of visits (stacked)', fontsize=13)
ax.set_title(
    f'{instrument} \u2014 Monthly visit counts per LSST band (stacked)',
    fontsize=14, fontweight='bold',
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda val, _: f'{int(val):,}'))
ax.legend(
    title='Band', title_fontsize=10,
    fontsize=10, loc='upper left',
    framealpha=0.85,
)
ax.grid(axis='y', alpha=0.35, linestyle='--')
ax.set_xlim(x[0] - 0.5, x[-1] + 0.5)

plt.tight_layout()
plt.show()

## 11. Band fraction per month

In [ ]:
# Normalised (fractional) stacked bar — useful to spot scheduling changes
fig, ax = plt.subplots(figsize=(max(10, len(months) * 1.6), 5))

totals_sum = np.array(
    [sum(monthly_counts.get(m, {}).values()) for m in month_labels],
    dtype=float,
)
totals_sum[totals_sum == 0] = np.nan   # avoid division by zero

bottom = np.zeros(len(month_labels))

for band in BANDS:
    counts = np.array([monthly_counts.get(m, {}).get(band, 0) for m in month_labels],
                      dtype=float)
    fractions = counts / totals_sum * 100.0
    fractions = np.nan_to_num(fractions)
    ax.bar(
        x,
        fractions,
        bottom=bottom,
        label=f'Band {band}',
        color=BAND_COLORS[band],
        edgecolor='white',
        linewidth=0.5,
        alpha=0.92,
    )
    bottom += fractions

ax.set_xticks(x)
ax.set_xticklabels(month_labels, rotation=30, ha='right', fontsize=11)
ax.set_xlabel('Month', fontsize=13)
ax.set_ylabel('Fraction of visits (%)', fontsize=13)
ax.set_ylim(0, 110)
ax.set_title(
    f'{instrument} \u2014 Band fraction per month (normalised)',
    fontsize=14, fontweight='bold',
)
ax.legend(
    title='Band', title_fontsize=10,
    fontsize=10, loc='upper right',
    framealpha=0.85,
)
ax.grid(axis='y', alpha=0.35, linestyle='--')
ax.set_xlim(x[0] - 0.5, x[-1] + 0.5)

plt.tight_layout()
plt.show()